<a href="https://colab.research.google.com/github/vamsiporeddy123/CSA6102-DIgital-Forencics/blob/main/Exp_29.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datetime import datetime, timedelta

def parse_time(t):
    return datetime.strptime(t, "%Y-%m-%d %H:%M:%S")

def detect_bruteforce(events, threshold=5, window_minutes=2):

    events = sorted(events, key=lambda e: parse_time(e["timestamp"]))

    by_account = {}
    for event in events:
        account = event["account"]
        if account not in by_account:
            by_account[account] = []
        by_account[account].append(event)

    results = {}

    for account, account_events in by_account.items():

        failures = [e for e in account_events if e["event_id"] == 4625]
        successes = [e for e in account_events if e["event_id"] == 4624]

        flagged = False

        for i in range(len(failures)):
            start = parse_time(failures[i]["timestamp"])
            end = start + timedelta(minutes=window_minutes)

            count = 0
            for failure in failures:
                t = parse_time(failure["timestamp"])
                if start <= t <= end:
                    count += 1

            if count >= threshold:
                flagged = True
                break

        if flagged:
            success_after = False
            if successes:
                last_failure = parse_time(failures[-1]["timestamp"])
                for success in successes:
                    if parse_time(success["timestamp"]) > last_failure:
                        success_after = True
                        break

            results[account] = {
                "failed_attempts": len(failures),
                "followed_by_success": success_after,
                "source_ips": sorted(set(f["source_ip"] for f in failures))
            }

    return results


events = [
    {
        "timestamp": "2026-08-04 09:00:00",
        "event_id": 4625,
        "account": "admin",
        "source_ip": "192.168.1.100"
    },
    {
        "timestamp": "2026-08-04 09:00:20",
        "event_id": 4625,
        "account": "admin",
        "source_ip": "192.168.1.100"
    },
    {
        "timestamp": "2026-08-04 09:00:40",
        "event_id": 4625,
        "account": "admin",
        "source_ip": "192.168.1.100"
    },
    {
        "timestamp": "2026-08-04 09:01:00",
        "event_id": 4625,
        "account": "admin",
        "source_ip": "192.168.1.100"
    },
    {
        "timestamp": "2026-08-04 09:01:20",
        "event_id": 4625,
        "account": "admin",
        "source_ip": "192.168.1.100"
    },
    {
        "timestamp": "2026-08-04 09:02:00",
        "event_id": 4624,
        "account": "admin",
        "source_ip": "192.168.1.100"
    },
    {
        "timestamp": "2026-08-04 10:00:00",
        "event_id": 4624,
        "account": "user1",
        "source_ip": "192.168.1.101"
    }
]


results = detect_bruteforce(events)


print("=" * 50)
print("BRUTE-FORCE DETECTION REPORT")
print("=" * 50)

if results:
    for account, details in results.items():
        print(f"\nAccount              : {account}")
        print(f"Failed Attempts      : {details['failed_attempts']}")
        print(f"Successful Login     : {details['followed_by_success']}")
        print(f"Source IP(s)         : {', '.join(details['source_ips'])}")
else:
    print("No brute-force attack detected.")

print("\nAnalysis Completed.")


BRUTE-FORCE DETECTION REPORT

Account              : admin
Failed Attempts      : 5
Successful Login     : True
Source IP(s)         : 192.168.1.100

Analysis Completed.
